In [ ]:
import os
import shutil
from pathlib import Path
from dotenv import load_dotenv
from minio import Minio
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta import configure_spark_with_delta_pip

In [ ]:
load_dotenv()

is_docker = os.path.exists("/.dockerenv")
MINIO_HOST = "minio:9000" if is_docker else "localhost:9000"
MINIO_USER = os.getenv("MINIO_ROOT_USER",     "minioadmin")
MINIO_PASS = os.getenv("MINIO_ROOT_PASSWORD", "minioadmin")

builder = (
    SparkSession.builder
    .appName("gold-dimensoes")
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.hadoop.fs.defaultFS", "file:///")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

cliente_minio = Minio(
    MINIO_HOST, access_key=MINIO_USER, secret_key=MINIO_PASS, secure=False
)

In [ ]:
buckets = {b.name for b in cliente_minio.list_buckets()}
print("Buckets:", buckets)
if "gold" not in buckets:
    cliente_minio.make_bucket("gold")
    print("Bucket 'gold' criado.")

In [ ]:
def download_delta_do_minio(bucket: str, prefix: str,
                            cliente: Minio, local_dir: Path) -> None:
    """Baixa toda a estrutura Delta (parquet + _delta_log) do MinIO para local."""
    if local_dir.exists():
        shutil.rmtree(local_dir)
    local_dir.mkdir(parents=True)
    objetos = list(cliente.list_objects(bucket, prefix=f"{prefix}/", recursive=True))
    print(f"  {len(objetos)} arquivo(s) em {bucket}/{prefix}/")
    for obj in objetos:
        relative = obj.object_name[len(prefix) + 1:]
        destino = local_dir / relative
        destino.parent.mkdir(parents=True, exist_ok=True)
        cliente.fget_object(bucket, obj.object_name, str(destino))


def upload_delta_para_minio(local_dir: Path, bucket: str, prefix: str,
                            cliente: Minio) -> None:
    """Sobe recursivamente a estrutura Delta para o MinIO."""
    for arq in sorted(local_dir.rglob("*")):
        if arq.is_file():
            object_name = f"{prefix}/{arq.relative_to(local_dir)}"
            cliente.fput_object(bucket, object_name, str(arq))
            print(f"  → {object_name}")

## dim_tempo



In [ ]:
print("\n" + "=" * 55)
print("Gold: dim_tempo")
print("=" * 55)

local_repro = Path("/tmp/silver_local/reproducoes")
download_delta_do_minio("silver", "fatos/reproducoes", cliente_minio, local_repro)
df_repro_datas = (
    spark.read.format("delta").load(str(local_repro))
    .select(F.to_date("timestamp").alias("data"))
)

local_pag = Path("/tmp/silver_local/pagamentos")
download_delta_do_minio("silver", "fatos/pagamentos", cliente_minio, local_pag)
df_pag_datas = (
    spark.read.format("delta").load(str(local_pag))
    .select(F.to_date("data").alias("data"))
)

data_min, data_max = (
    df_repro_datas.union(df_pag_datas)
    .agg(F.min("data"), F.max("data"))
    .first()
)
print(f"Calendário: {data_min} até {data_max}")

In [ ]:
DIAS_SEMANA = [
    "Domingo", "Segunda-feira", "Terça-feira", "Quarta-feira",
    "Quinta-feira", "Sexta-feira", "Sábado",
]
array_dias_semana = F.array(*[F.lit(d) for d in DIAS_SEMANA])

df_tempo = (
    spark.sql(f"""
        SELECT explode(sequence(to_date('{data_min}'), to_date('{data_max}'), interval 1 day)) AS data
    """)
    .withColumn("data_id", F.date_format("data", "yyyyMMdd").cast("int"))
    .withColumn("ano", F.year("data"))
    .withColumn("mes", F.month("data"))
    .withColumn("dia", F.dayofmonth("data"))
    # dayofweek do Spark: 1=Domingo..7=Sábado — alinhado ao índice da lista acima
    .withColumn("dia_semana", F.element_at(array_dias_semana, F.dayofweek("data")))
    .withColumn("ano_mes", F.date_format("data", "yyyy-MM"))
    .select("data_id", "data", "ano", "mes", "dia", "dia_semana", "ano_mes")
    .orderBy("data")
)

print(f"dim_tempo gerada: {df_tempo.count():,} dias")

local_gold = Path("/tmp/gold/dim_tempo")
if local_gold.exists():
    shutil.rmtree(local_gold)

df_tempo.write.format("delta").mode("overwrite").save(str(local_gold))
print(f"Gold Delta escrito em {local_gold}")

upload_delta_para_minio(local_gold, "gold", "dimensoes/dim_tempo", cliente_minio)
print("dim_tempo concluído.")

## dim_usuario

In [ ]:
print("\n" + "=" * 55)
print("Gold: dim_usuario")
print("=" * 55)

local_silver = Path("/tmp/silver_local/usuarios")
download_delta_do_minio("silver", "dimensoes/usuarios", cliente_minio, local_silver)

df_usuario = (
    spark.read.format("delta").load(str(local_silver))
    .select(
        F.col("id").alias("usuario_id"),
        "nome", "pais", "data_cadastro",
    )
)

local_gold = Path("/tmp/gold/dim_usuario")
if local_gold.exists():
    shutil.rmtree(local_gold)

df_usuario.write.format("delta").mode("overwrite").save(str(local_gold))
upload_delta_para_minio(local_gold, "gold", "dimensoes/dim_usuario", cliente_minio)
print(f"dim_usuario concluído — {df_usuario.count():,} registros.")

## dim_plano

In [ ]:
print("\n" + "=" * 55)
print("Gold: dim_plano")
print("=" * 55)

local_silver = Path("/tmp/silver_local/planos")
download_delta_do_minio("silver", "dimensoes/planos", cliente_minio, local_silver)

df_plano = (
    spark.read.format("delta").load(str(local_silver))
    .select(
        F.col("id").alias("plano_id"),
        "nome", "preco_mensal",
    )
)

local_gold = Path("/tmp/gold/dim_plano")
if local_gold.exists():
    shutil.rmtree(local_gold)

df_plano.write.format("delta").mode("overwrite").save(str(local_gold))
upload_delta_para_minio(local_gold, "gold", "dimensoes/dim_plano", cliente_minio)
print(f"dim_plano concluído — {df_plano.count():,} registros.")

## dim_artista

In [ ]:
print("\n" + "=" * 55)
print("Gold: dim_artista")
print("=" * 55)

local_silver = Path("/tmp/silver_local/artistas")
download_delta_do_minio("silver", "dimensoes/artistas", cliente_minio, local_silver)

df_artista = (
    spark.read.format("delta").load(str(local_silver))
    .select(
        F.col("id").alias("artista_id"),
        "nome", "pais",
    )
)

local_gold = Path("/tmp/gold/dim_artista")
if local_gold.exists():
    shutil.rmtree(local_gold)

df_artista.write.format("delta").mode("overwrite").save(str(local_gold))
upload_delta_para_minio(local_gold, "gold", "dimensoes/dim_artista", cliente_minio)
print(f"dim_artista concluído — {df_artista.count():,} registros.")

## dim_musica



In [ ]:
print("\n" + "=" * 55)
print("Gold: dim_musica")
print("=" * 55)

local_musicas = Path("/tmp/silver_local/musicas")
download_delta_do_minio("silver", "dimensoes/musicas", cliente_minio, local_musicas)
df_musicas = spark.read.format("delta").load(str(local_musicas))

local_albuns = Path("/tmp/silver_local/albuns")
download_delta_do_minio("silver", "dimensoes/albuns", cliente_minio, local_albuns)
df_albuns = (
    spark.read.format("delta").load(str(local_albuns))
    .select(F.col("id").alias("album_id"), F.col("titulo").alias("album"))
)

local_generos = Path("/tmp/silver_local/generos")
download_delta_do_minio("silver", "dimensoes/generos", cliente_minio, local_generos)
df_generos = (
    spark.read.format("delta").load(str(local_generos))
    .select(F.col("id").alias("genero_id"), F.col("nome").alias("genero"))
)

df_musica = (
    df_musicas
    .join(df_albuns, on="album_id", how="left")
    .join(df_generos, on="genero_id", how="left")
    .select(
        F.col("id").alias("musica_id"),
        "titulo", "genero", "album", "artista_id", "duracao_ms",
    )
)

local_gold = Path("/tmp/gold/dim_musica")
if local_gold.exists():
    shutil.rmtree(local_gold)

df_musica.write.format("delta").mode("overwrite").save(str(local_gold))
upload_delta_para_minio(local_gold, "gold", "dimensoes/dim_musica", cliente_minio)
print(f"dim_musica concluído — {df_musica.count():,} registros.")
print("\nTransformação Gold de todas as dimensões concluída.")

In [ ]:
spark.stop()